# PGx Risk Calculator Dashboard – Full Deployment Workflow

**Purpose:** Deploy the PGx Risk Calculator Dashboard from cohorts with aggregated feature importances through Lambda/Docker.  
**Updated:** January 2026  
**Model mapping (from PHTS):** Baseline → **opioid_ed** (predictive model); Extended → **polypharmacy** (non_opioid_ed model).

## Overview

This notebook implements the **full risk calculator dashboard deployment workflow** for PGx, starting from:

- **Cohorts with aggregated feature importances** (Step 3 `aggregated_feature_importance`, Step 3b `cohort_feature_importance`)
- **Final models** (Step 6 `6_final_model/outputs`)
- **SHAP/FFA outputs** (Steps 7 & 8, optional for causal tab)

## PGx Model Mapping (PHTS → PGx)

| PHTS | PGx cohort | Age bands | Description |
|------|------------|-----------|-------------|
| **Baseline** | `opioid_ed` | 13-24, 25-44, 45-54, 55-64 | Opioid-related ED visit predictive model |
| **Extended** | `non_opioid_ed` (polypharmacy) | 65-74, 75-84, 85-94 | Polypharmacy / adverse drug event model |

## Workflow Steps

1. **Verify inputs** – Cohorts and aggregated feature importance (Step 3/3b), final models (Step 6).
2. **Generate metadata** – Extract valid codes from feature importance for dashboard dropdowns.
3. **Prepare models** – Package models and feature schemas from `6_final_model/outputs`.
4. **Combine SHAP/FFA** (optional) – For causal analysis tab.
5. **Prepare Lambda directory** – Assemble `lambda_dir` for Docker build.
6. **Verify & deploy** – Verify `lambda_dir`, then build Docker image and deploy (ECR/API Gateway).

## Reference

- PHTS calculator workflow: `C:\Projects\phts\graft-loss\cohort_analysis\calculator\calculator_workflow.ipynb`
- PHTS scripts: `C:\Projects\phts\scripts`
- PGx data prep: `9_risk_dashboard/data_preparation/`

In [ ]:
# Setup: paths and project root
import sys
import subprocess
from pathlib import Path

PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == "9_risk_dashboard":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif (PROJECT_ROOT / "9_risk_dashboard").exists():
    pass
else:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent

DASHBOARD_DIR = PROJECT_ROOT / "9_risk_dashboard"
DATA_PREP_DIR = DASHBOARD_DIR / "data_preparation"
DEPLOY_DIR = DASHBOARD_DIR / "deployment"
sys.path.insert(0, str(PROJECT_ROOT))

print("PGx Risk Calculator Workflow")
print("=" * 60)
print(f"Project root: {PROJECT_ROOT}")
print(f"Dashboard dir: {DASHBOARD_DIR}")
print(f"Data prep: {DATA_PREP_DIR}")
print("=" * 60)

In [ ]:
# Configuration: PGx cohorts and age bands (aligned with prepare_lambda_dir.py)
# Baseline → opioid_ed; Extended → non_opioid_ed (polypharmacy)
REQUIRED_COHORTS = {
    "opioid_ed": ["13-24", "25-44", "45-54", "55-64"],
    "non_opioid_ed": ["65-74", "75-84", "85-94"],
}

STEP3_OUTPUTS = PROJECT_ROOT / "3_feature_importance" / "outputs"
STEP3B_OUTPUTS = PROJECT_ROOT / "3b_feature_importance_eda" / "outputs"
FINAL_MODEL_OUTPUTS = PROJECT_ROOT / "6_final_model" / "outputs"

print("Cohorts and age bands:")
for cohort, bands in REQUIRED_COHORTS.items():
    print(f"  {cohort}: {bands}")
print("\nInput dirs:")
print(f"  Step 3:   {STEP3_OUTPUTS}")
print(f"  Step 3b:  {STEP3B_OUTPUTS}")
print(f"  Step 6:   {FINAL_MODEL_OUTPUTS}")

## Step 0: Verify cohorts and aggregated feature importances

Ensure feature importance (Step 3 or 3b) and final models (Step 6) exist for each cohort/age_band.

In [ ]:
def check_feature_importance(cohort: str, age_band: str) -> bool:
    ab = age_band.replace("-", "_")
    # Step 3b refined
    fi_3b = STEP3B_OUTPUTS / cohort / ab / f"{cohort}_{ab}_cohort_feature_importance.csv"
    if fi_3b.exists():
        return True
    # Step 3 aggregated
    fi_3 = STEP3_OUTPUTS / cohort / ab / f"{cohort}_{ab}_aggregated_feature_importance.csv"
    return fi_3.exists()

def check_final_model(cohort: str, age_band: str) -> bool:
    ab = age_band.replace("-", "_")
    model_dir = FINAL_MODEL_OUTPUTS / cohort / ab
    if not model_dir.exists():
        return False
    # At least feature schema or one model file
    models_sub = model_dir / "models"
    if models_sub.exists():
        return any(models_sub.glob("*.joblib")) or (model_dir / "feature_schema.json").exists()
    return (model_dir / "feature_schema.json").exists()

print("Step 0: Verifying inputs per cohort/age_band\n")
all_ok = True
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        fi_ok = check_feature_importance(cohort, age_band)
        model_ok = check_final_model(cohort, age_band)
        status = "OK" if (fi_ok and model_ok) else "MISSING"
        if not (fi_ok and model_ok):
            all_ok = False
        print(f"  {cohort} / {age_band}:  FI={fi_ok}, Model={model_ok}  -> {status}")
print("\n" + ("All inputs present." if all_ok else "Fix missing inputs before running data preparation."))

## Step 1: Generate metadata

Extract valid codes (drugs, ICD, CPT) from feature importance for dashboard dropdowns. Uses Step 3b `cohort_feature_importance` when available, else Step 3 `aggregated_feature_importance`.

In [ ]:
subprocess.run([sys.executable, "generate_metadata.py", "--all"], cwd=DATA_PREP_DIR, check=True)

## Step 2: Prepare models

Package models and feature schemas from `6_final_model/outputs` into `9_risk_dashboard/outputs/models`.

In [ ]:
subprocess.run([sys.executable, "prepare_models.py", "--all"], cwd=DATA_PREP_DIR, check=True)

## Step 3 (optional): Combine SHAP and FFA results

For the Causal Analysis tab, combine SHAP (Step 7) and FFA (Step 8) results. Run per cohort/age_band if you have those outputs.

In [ ]:
# Optional: run for one or all cohort/age_band
# !python combine_shap_ffa_results.py --cohort opioid_ed --age-band 25-44 --output-dir "$PROJECT_ROOT/9_risk_dashboard/outputs"
# For all: implement loop or use --all-cohorts if supported
print("Optional: run combine_shap_ffa_results.py for cohorts that have SHAP/FFA outputs.")

## Step 4: Prepare Lambda directory

Assemble `lambda_dir` under `9_risk_dashboard` for Docker build (models, metadata, CPIC data).

In [ ]:
%cd "$DEPLOY_DIR"
!python prepare_lambda_dir.py

## Step 5: Verify Lambda directory

Ensure all required files are present before building the image.

In [ ]:
subprocess.run([sys.executable, "prepare_lambda_dir.py", "--verify-only"], cwd=DEPLOY_DIR, check=True)

## Step 6: Build and deploy

Build the Docker image and push to ECR; then update API Gateway/Lambda. Use the deployment script in `9_risk_dashboard/deployment`.

In [ ]:
# From 9_risk_dashboard directory:
# ./deployment/docker_build.sh
# Or manually:
# docker build -t pgx-risk-dashboard .
# Then push to ECR and update Lambda function (see docs/Step10_Results/README_results_deployment.md)
print("Run from shell: cd 9_risk_dashboard && ./deployment/docker_build.sh")
print("See: docs/Step10_Results/README_results_deployment.md")